<a href="https://colab.research.google.com/github/MoneebAliKarrar/Machine_Translation/blob/main/MachineTranslation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Imports

In [ ]:
import pandas as pd
import torch
import numpy as np
import os
import random
!pip install sentencepiece

In [ ]:
!pip install --upgrade pip
!pip install "transformers>=4.40.0" "datasets>=4.0.0" sentencepiece accelerate


In [ ]:
import torch, random, numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


Checking MY GPU

In [ ]:
#import torch
#torch.cuda.get_device_name(0)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
arabic_path = "/content/drive/MyDrive/translation_data/arbnav/arabic.txt"
igbo_path   = "/content/drive/MyDrive/translation_data/igbo/ibo.txt"

In [ ]:
import os, re, unicodedata, json
from collections import Counter, OrderedDict
import numpy as np

# =========================
# Config
# =========================
AR_FOLDER = "/content/drive/MyDrive/translation_data/arbnav"
IG_FOLDER = "/content/drive/MyDrive/translation_data/igbo"
OUT_DIR   = "/content/drive/MyDrive/translation_data/audit"
os.makedirs(OUT_DIR, exist_ok=True)

NOISY_RATIO_THRESH = 0.55

# learn repeated title/heading pairs from early aligned pairs
HEADING_SCAN_PAIRS = 50
HEADING_MIN_FREQ   = 2

# drop title-like pairs if they appear very early in a chapter
FRONT_TITLE_PAIRS = 6

# repair mismatched raw chapters (safe)
REPAIR_RAW_MISMATCH = True
REPAIR_MAX_REMOVALS = 3
REPAIR_SEARCH_WINDOW = 25

BOOK_CHAPTER_RE = re.compile(r'_([0-9A-Z]{2,3})_(\d{1,3})_read\.txt$', re.IGNORECASE)
ALL_DIGITS_BOOK = re.compile(r"^\d+$")

# Arabic-Indic digit support
DIGIT_MAP = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

# =========================
# File indexing / reading
# =========================
def get_book_chapter(fname):
    m = BOOK_CHAPTER_RE.search(fname)
    if not m:
        return None
    return f"{m.group(1)}_{int(m.group(2))}"

def index_folder(folder):
    d = {}
    collisions = Counter()
    for fn in os.listdir(folder):
        key = get_book_chapter(fn)
        if key:
            if key in d:
                collisions[key] += 1
            d[key] = os.path.join(folder, fn)
    return OrderedDict(sorted(d.items(), key=lambda x: x[0])), collisions

def read_lines(path):
    # only drop empty lines to preserve alignment
    with open(path, encoding="utf-8") as f:
        return [l.rstrip("\n") for l in f if l.strip()]

# =========================
# Normalization
# =========================
BOM = "\ufeff"

def strip_format_chars(s: str) -> str:
    # Remove RTL/LTR marks etc. (Unicode category 'Cf')
    return "".join(ch for ch in s if unicodedata.category(ch) != "Cf")

def normalize_arabic(text):
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)
    text = re.sub("[ًٌٍَُِّْـ]", "", text)  # tashkeel
    return text

def normalize_text(text, lang):
    text = text.lstrip(BOM)
    text = strip_format_chars(text)
    text = text.translate(DIGIT_MAP)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    if lang == "ar":
        text = normalize_arabic(text)
    return text

# =========================
# Noise detection
# =========================
AR_LETTERS = re.compile(r"[\u0600-\u06FF]")
LATIN_LETTERS = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿƁɓƊɗẸẹẹỊịỌọỤụÑñ]")
NON_SPACE = re.compile(r"\S")

def letter_ratio(text, lang):
    if not text:
        return 0.0
    non_space = len(NON_SPACE.findall(text))
    if non_space == 0:
        return 0.0
    letters = len(AR_LETTERS.findall(text)) if lang == "ar" else len(LATIN_LETTERS.findall(text))
    return letters / non_space

def is_noisy(text, lang, ratio_thresh=0.55):
    if not text:
        return True
    if "\ufffd" in text:
        return True
    if len(text) < 2:
        return True
    if letter_ratio(text, lang) < ratio_thresh:
        return True
    return False

# =========================
# Structural rules
# =========================
NUM_ONLY = re.compile(r"^[0-9]{1,3}\.?$")

# IMPORTANT: tolerant leading-number parser (works even with no space)
LEAD_NUM = re.compile(r"^\s*([0-9]{1,3})[.)]?\s*(.*?)\s*$")

PUNCT = set(list(".,!?؛،:؟\"'“”‘’…()[]{}<>"))

def has_any_punct(s: str) -> bool:
    return any(ch in PUNCT for ch in s)

def is_speaker_tag(t: str) -> bool:
    # "ايوب:" / "Job sịrị:"
    return t.endswith(":") and len(t) <= 30 and len(t.split()) <= 4

def is_name_label_after_number(rest: str) -> bool:
    # drops: "ايوب" "اليفاز" (short labels) after a leading number
    if not rest:
        return True
    if has_any_punct(rest):
        return False
    if len(rest.split()) <= 2 and len(rest) <= 20:
        return True
    return False

def strip_tail_punct(s: str) -> str:
    return s.strip(" .،؛:!?؟\"'“”‘’…")

def heading_key(text: str, lang: str) -> str:
    """
    normalize candidate:
    - normalize_text
    - if starts with number, remove it
    - strip trailing punctuation
    - lowercase for ig
    """
    t = normalize_text(text, lang)
    if not t:
        return ""
    if NUM_ONLY.match(t):
        return ""

    m = LEAD_NUM.match(t)
    if m and m.group(2):
        t = m.group(2).strip()

    t = strip_tail_punct(t)
    t = re.sub(r"\s+", " ", t).strip()
    if lang != "ar":
        t = t.lower()
    return t

def is_ar_title_candidate(ka: str) -> bool:
    # Arabic title must be a SINGLE token (no spaces)
    if not ka:
        return False
    if " " in ka:
        return False
    if len(ka) > 25:
        return False
    if any(ch.isdigit() for ch in ka):
        return False
    return True

def is_ig_title_candidate(ki: str) -> bool:
    if not ki:
        return False
    if len(ki) > 35:
        return False
    if len(ki.split()) > 4:   # allows "ọrụ ndị ozi"
        return False
    if any(ch.isdigit() for ch in ki):
        return False
    return True

def parse_line(line: str, lang: str):
    """
    Returns:
      text: cleaned text (leading number stripped if present)
      drop_reason: reason or None
      hk: heading_key
      title_cand: bool (language-specific)
    """
    t = normalize_text(line, lang)
    if not t:
        return {"text":"", "drop_reason":"empty", "hk":"", "title_cand":False}

    if NUM_ONLY.match(t):
        return {"text":"", "drop_reason":"num_only", "hk":"", "title_cand":False}

    if is_speaker_tag(t):
        return {"text":"", "drop_reason":"speaker_tag", "hk":"", "title_cand":False}

    # strip leading number if exists
    m = LEAD_NUM.match(t)
    if m and m.group(2):
        rest = m.group(2).strip()
        # name label like "4 ايوب"
        if is_name_label_after_number(rest) and lang == "ar":
            return {"text":"", "drop_reason":"name_label", "hk":"", "title_cand":False}
        t = rest

    if is_noisy(t, lang, NOISY_RATIO_THRESH):
        return {"text":"", "drop_reason":"noisy", "hk":"", "title_cand":False}

    hk = heading_key(t, lang)
    if lang == "ar":
        tc = is_ar_title_candidate(hk)
    else:
        tc = is_ig_title_candidate(hk)

    return {"text":t, "drop_reason":None, "hk":hk, "title_cand":tc}

# =========================
# Repair raw mismatches (safe)
# =========================
def is_repair_drop_candidate(raw_line: str, lang: str) -> bool:
    t = normalize_text(raw_line, lang)
    if not t:
        return True
    if NUM_ONLY.match(t):
        return True
    if is_speaker_tag(t):
        return True

    m = LEAD_NUM.match(t)
    if m and m.group(2):
        rest = m.group(2).strip()
        # numbered short labels/titles are safe drops
        rk = heading_key(rest, lang)
        if lang == "ar" and is_ar_title_candidate(rk):
            return True
        if lang != "ar" and is_ig_title_candidate(rk):
            return True
        if lang == "ar" and is_name_label_after_number(rest):
            return True

    # also allow very short standalone titles at boundaries
    rk = heading_key(t, lang)
    if lang == "ar" and is_ar_title_candidate(rk):
        return True
    if lang != "ar" and is_ig_title_candidate(rk):
        return True

    return False

def repair_to_equal_length(ar_raw, ig_raw):
    ar = list(ar_raw)
    ig = list(ig_raw)

    for _ in range(REPAIR_MAX_REMOVALS):
        if len(ar) == len(ig):
            return ar, ig, True

        if len(ar) > len(ig):
            removed = False
            for idx in range(min(REPAIR_SEARCH_WINDOW, len(ar))):
                if is_repair_drop_candidate(ar[idx], "ar"):
                    ar.pop(idx); removed = True; break
            if not removed:
                for idx in range(len(ar)-1, max(-1, len(ar)-REPAIR_SEARCH_WINDOW-1), -1):
                    if is_repair_drop_candidate(ar[idx], "ar"):
                        ar.pop(idx); removed = True; break
            if not removed:
                break
        else:
            removed = False
            for idx in range(min(REPAIR_SEARCH_WINDOW, len(ig))):
                if is_repair_drop_candidate(ig[idx], "ig"):
                    ig.pop(idx); removed = True; break
            if not removed:
                for idx in range(len(ig)-1, max(-1, len(ig)-REPAIR_SEARCH_WINDOW-1), -1):
                    if is_repair_drop_candidate(ig[idx], "ig"):
                        ig.pop(idx); removed = True; break
            if not removed:
                break

    return ar, ig, (len(ar) == len(ig))

# =========================
# Build heading-pair set (strict)
# =========================
def build_heading_pair_set(ar_files, ig_files, common_keys):
    pair_counts = Counter()

    for key in common_keys:
        book = key.split("_")[0]
        if ALL_DIGITS_BOOK.match(book):
            continue

        ar_raw = read_lines(ar_files[key])
        ig_raw = read_lines(ig_files[key])
        if len(ar_raw) != len(ig_raw):
            continue

        for a_line, i_line in list(zip(ar_raw, ig_raw))[:HEADING_SCAN_PAIRS]:
            pa = parse_line(a_line, "ar")
            pi = parse_line(i_line, "ig")
            if pa["drop_reason"] or pi["drop_reason"]:
                continue

            if pa["title_cand"] and pi["title_cand"]:
                pair_counts[(pa["hk"], pi["hk"])] += 1

    heading_pairs = {p for p,c in pair_counts.items() if c >= HEADING_MIN_FREQ}

    with open(os.path.join(OUT_DIR, "heading_pair_candidates.json"), "w", encoding="utf-8") as f:
        json.dump([{"ar":k[0], "ig":k[1], "count":c} for k,c in pair_counts.most_common(800)],
                  f, ensure_ascii=False, indent=2)
    with open(os.path.join(OUT_DIR, "heading_pair_set.json"), "w", encoding="utf-8") as f:
        json.dump([{"ar":k[0], "ig":k[1]} for k in sorted(list(heading_pairs))],
                  f, ensure_ascii=False, indent=2)

    return heading_pairs

# =========================
# Reporting
# =========================
def summarize_lengths(arr):
    return {
        "min": int(arr.min()) if len(arr) else 0,
        "avg": float(arr.mean()) if len(arr) else 0.0,
        "max": int(arr.max()) if len(arr) else 0,
        "p50": float(np.percentile(arr, 50)) if len(arr) else 0.0,
        "p95": float(np.percentile(arr, 95)) if len(arr) else 0.0,
        "p99": float(np.percentile(arr, 99)) if len(arr) else 0.0,
    }

# =========================
# Audit (pairwise AFTER alignment)
# =========================
def audit_dataset_pairwise_v3():
    ar_files, ar_collisions = index_folder(AR_FOLDER)
    ig_files, ig_collisions = index_folder(IG_FOLDER)

    ar_keys = set(ar_files.keys())
    ig_keys = set(ig_files.keys())
    common_keys = sorted(list(ar_keys & ig_keys))

    missing_in_ig = sorted(list(ar_keys - ig_keys))
    missing_in_ar = sorted(list(ig_keys - ar_keys))

    heading_pairs = build_heading_pair_set(ar_files, ig_files, common_keys)

    mismatches = []
    repaired_mismatch_chapters = 0

    kept_ar, kept_ig = [], []
    removed_counts = Counter()

    for key in common_keys:
        book = key.split("_")[0]
        if ALL_DIGITS_BOOK.match(book):
            removed_counts["skipped_preface_like"] += 1
            continue

        ar_raw = read_lines(ar_files[key])
        ig_raw = read_lines(ig_files[key])

        if len(ar_raw) != len(ig_raw) and REPAIR_RAW_MISMATCH:
            ar_fixed, ig_fixed, ok = repair_to_equal_length(ar_raw, ig_raw)
            if ok:
                repaired_mismatch_chapters += 1
                ar_raw, ig_raw = ar_fixed, ig_fixed

        if len(ar_raw) != len(ig_raw):
            mismatches.append({
                "key": key,
                "ar_verses": len(ar_raw),
                "ig_verses": len(ig_raw),
                "reason": "raw_verse_count_mismatch"
            })
            continue

        for idx, (a_line, i_line) in enumerate(zip(ar_raw, ig_raw), start=1):
            pa = parse_line(a_line, "ar")
            pi = parse_line(i_line, "ig")

            if pa["drop_reason"]:
                removed_counts[pa["drop_reason"]] += 1
                continue
            if pi["drop_reason"]:
                removed_counts[pi["drop_reason"]] += 1
                continue

            # drop learned heading pairs anywhere
            if (pa["hk"], pi["hk"]) in heading_pairs:
                removed_counts["paired_heading"] += 1
                continue

            # drop title-like pairs ONLY at the start of a chapter
            if idx <= FRONT_TITLE_PAIRS and pa["title_cand"] and pi["title_cand"]:
                removed_counts["front_title_pair"] += 1
                continue

            kept_ar.append(pa["text"])
            kept_ig.append(pi["text"])

    total_pairs_kept = len(kept_ar)
    empty_after_norm = sum(1 for a,i in zip(kept_ar, kept_ig) if (not a.strip()) or (not i.strip()))

    pair_counter = Counter(zip(kept_ar, kept_ig))
    dup_pairs = sum(1 for _,v in pair_counter.items() if v > 1)
    dup_pairs_instances = sum(v-1 for v in pair_counter.values() if v > 1)
    dup_src = sum(1 for v in Counter(kept_ar).values() if v > 1)
    dup_tgt = sum(1 for v in Counter(kept_ig).values() if v > 1)

    def word_count(s): return len(s.split()) if s else 0
    ar_char_lens = np.array([len(s) for s in kept_ar], dtype=int)
    ig_char_lens = np.array([len(s) for s in kept_ig], dtype=int)
    ar_word_lens = np.array([word_count(s) for s in kept_ar], dtype=int)
    ig_word_lens = np.array([word_count(s) for s in kept_ig], dtype=int)

    report = {
        "files": {
            "arabic_chapter_files": len(ar_files),
            "igbo_chapter_files": len(ig_files),
            "common_keys": len(common_keys),
            "missing_in_arabic": len(missing_in_ar),
            "missing_in_igbo": len(missing_in_ig),
            "mismatched_chapters_raw": len(mismatches),
            "repaired_mismatch_chapters": repaired_mismatch_chapters,
            "arabic_key_collisions": int(sum(ar_collisions.values())),
            "igbo_key_collisions": int(sum(ig_collisions.values())),
        },
        "heading_pairs": {
            "paired_heading_set_size": len(heading_pairs),
            "heading_scan_pairs": HEADING_SCAN_PAIRS,
            "heading_min_freq": HEADING_MIN_FREQ,
            "front_title_pairs": FRONT_TITLE_PAIRS
        },
        "removed_counts_pairwise": dict(removed_counts),
        "kept": {
            "total_verse_pairs_kept": total_pairs_kept,
            "empty_after_normalization_pairs": empty_after_norm,
            "noisy_ratio_threshold": NOISY_RATIO_THRESH
        },
        "duplicates": {
            "duplicate_unique_pairs": dup_pairs,
            "duplicate_pair_extra_instances": dup_pairs_instances,
            "duplicate_unique_ar_sentences": dup_src,
            "duplicate_unique_ig_sentences": dup_tgt
        },
        "lengths": {
            "arabic_chars": summarize_lengths(ar_char_lens),
            "igbo_chars": summarize_lengths(ig_char_lens),
            "arabic_words": summarize_lengths(ar_word_lens),
            "igbo_words": summarize_lengths(ig_word_lens)
        }
    }

    print("===== DATASET EVALUATION REPORT (PAIRWISE AFTER ALIGNMENT v3) =====")
    print(json.dumps(report, indent=2, ensure_ascii=False))

    print("\nTop 15 mismatches (raw count):")
    for m in mismatches[:15]:
        print(m)

    with open(os.path.join(OUT_DIR, "dataset_report_pairwise_v3.json"), "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    with open(os.path.join(OUT_DIR, "mismatches_pairwise_v3.json"), "w", encoding="utf-8") as f:
        json.dump(mismatches, f, ensure_ascii=False, indent=2)

    print(f"\n✅ Saved audit files to: {OUT_DIR}")

    print("\nShortest 20 kept pairs:")
    idxs = np.argsort([len(s) for s in kept_ar])[:20]
    for j in idxs:
        print(len(kept_ar[j]), kept_ar[j], "||", kept_ig[j])

    return report

report = audit_dataset_pairwise_v3()


In [ ]:
import os, re, unicodedata, json, random
from collections import Counter, OrderedDict

# =========================
# Paths
# =========================
AR_FOLDER = "/content/drive/MyDrive/translation_data/arbnav"
IG_FOLDER = "/content/drive/MyDrive/translation_data/igbo"
OUT_DIR   = "/content/drive/MyDrive/translation_data/processed_lenfilter"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# Split config
# =========================
SEED = 42
TRAIN_RATIO = 0.98
VALID_RATIO = 0.01
TEST_RATIO  = 0.01

assert abs((TRAIN_RATIO + VALID_RATIO + TEST_RATIO) - 1.0) < 1e-9

# =========================
# Repair config (keeps alignment; small safe repair)
# =========================
REPAIR_RAW_MISMATCH = True
REPAIR_MAX_REMOVALS = 3
REPAIR_SEARCH_WINDOW = 25

# =========================
# Option A filters
# =========================
MIN_AR_CHARS = 8
MIN_WORDS_EITHER_SIDE = 3   # i.e., drop if <=2 words

# =========================
# Helpers
# =========================
BOOK_CHAPTER_RE = re.compile(r'_([0-9A-Z]{2,3})_(\d{1,3})_read\.txt$', re.IGNORECASE)
ALL_DIGITS_BOOK = re.compile(r"^\d+$")

DIGIT_MAP = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
BOM = "\ufeff"

# tolerant leading-number parser (works with/without spaces)
LEAD_NUM = re.compile(r"^\s*([0-9]{1,3})[.)]?\s*(.*?)\s*$")
NUM_ONLY = re.compile(r"^[0-9]{1,3}\.?$")

def get_book_chapter(fname):
    m = BOOK_CHAPTER_RE.search(fname)
    if not m:
        return None
    return f"{m.group(1)}_{int(m.group(2))}"

def index_folder(folder):
    d = {}
    for fn in os.listdir(folder):
        key = get_book_chapter(fn)
        if key:
            d[key] = os.path.join(folder, fn)
    return OrderedDict(sorted(d.items(), key=lambda x: x[0]))

def read_lines(path):
    # keep alignment stable: only drop empty lines
    with open(path, encoding="utf-8") as f:
        return [l.rstrip("\n") for l in f if l.strip()]

def strip_format_chars(s: str) -> str:
    # remove RTL/LTR marks etc. (category Cf)
    return "".join(ch for ch in s if unicodedata.category(ch) != "Cf")

def normalize_arabic(text):
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)
    text = re.sub("[ًٌٍَُِّْـ]", "", text)  # tashkeel
    return text

def normalize_text(text, lang):
    text = text.lstrip(BOM)
    text = strip_format_chars(text)
    text = text.translate(DIGIT_MAP)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    if lang == "ar":
        text = normalize_arabic(text)
    return text

def strip_leading_number(t: str) -> str:
    m = LEAD_NUM.match(t)
    if m and m.group(2):
        return m.group(2).strip()
    return t

def is_repair_drop_candidate(raw_line: str, lang: str) -> bool:
    # safe drops only
    t = normalize_text(raw_line, lang)
    if not t:
        return True
    if NUM_ONLY.match(t):
        return True
    # also drop very short title-ish lines that often cause off-by-1
    rest = strip_leading_number(t)
    if len(rest.split()) <= 2 and len(rest) <= 25:
        return True
    return False

def repair_to_equal_length(ar_raw, ig_raw):
    ar = list(ar_raw)
    ig = list(ig_raw)

    for _ in range(REPAIR_MAX_REMOVALS):
        if len(ar) == len(ig):
            return ar, ig, True

        if len(ar) > len(ig):
            removed = False
            for idx in range(min(REPAIR_SEARCH_WINDOW, len(ar))):
                if is_repair_drop_candidate(ar[idx], "ar"):
                    ar.pop(idx); removed = True; break
            if not removed:
                for idx in range(len(ar)-1, max(-1, len(ar)-REPAIR_SEARCH_WINDOW-1), -1):
                    if is_repair_drop_candidate(ar[idx], "ar"):
                        ar.pop(idx); removed = True; break
            if not removed:
                break
        else:
            removed = False
            for idx in range(min(REPAIR_SEARCH_WINDOW, len(ig))):
                if is_repair_drop_candidate(ig[idx], "ig"):
                    ig.pop(idx); removed = True; break
            if not removed:
                for idx in range(len(ig)-1, max(-1, len(ig)-REPAIR_SEARCH_WINDOW-1), -1):
                    if is_repair_drop_candidate(ig[idx], "ig"):
                        ig.pop(idx); removed = True; break
            if not removed:
                break

    return ar, ig, (len(ar) == len(ig))

# =========================
# Build dataset (aligned -> filter -> dedup)
# =========================
ar_files = index_folder(AR_FOLDER)
ig_files = index_folder(IG_FOLDER)

common_keys = sorted(set(ar_files.keys()) & set(ig_files.keys()))

kept_pairs = []
stats = Counter()

mismatches = []
repaired = 0

for key in common_keys:
    book = key.split("_")[0]
    if ALL_DIGITS_BOOK.match(book):
        stats["skipped_preface_like"] += 1
        continue

    ar_raw = read_lines(ar_files[key])
    ig_raw = read_lines(ig_files[key])

    if len(ar_raw) != len(ig_raw) and REPAIR_RAW_MISMATCH:
        ar_fix, ig_fix, ok = repair_to_equal_length(ar_raw, ig_raw)
        if ok:
            repaired += 1
            ar_raw, ig_raw = ar_fix, ig_fix

    if len(ar_raw) != len(ig_raw):
        mismatches.append({"key": key, "ar": len(ar_raw), "ig": len(ig_raw)})
        continue

    # pairwise normalization + Option A filter
    for a_line, i_line in zip(ar_raw, ig_raw):
        a = normalize_text(a_line, "ar")
        i = normalize_text(i_line, "ig")

        # strip leading verse numbers like "10 ..." from both sides
        a = strip_leading_number(a)
        i = strip_leading_number(i)

        if not a or not i:
            stats["drop_empty"] += 1
            continue
        if NUM_ONLY.match(a) or NUM_ONLY.match(i):
            stats["drop_num_only"] += 1
            continue

        # Option A (simple length-based)
        if len(a.split()) < MIN_WORDS_EITHER_SIDE or len(i.split()) < MIN_WORDS_EITHER_SIDE:
            stats["drop_short_words"] += 1
            continue
        if len(a) < MIN_AR_CHARS:
            stats["drop_short_ar_chars"] += 1
            continue

        kept_pairs.append((a, i))
        stats["kept_before_dedup"] += 1

# Deduplicate exact pairs
seen = set()
deduped = []
for a, i in kept_pairs:
    if (a, i) in seen:
        stats["drop_duplicate_pair"] += 1
        continue
    seen.add((a, i))
    deduped.append((a, i))
stats["kept_after_dedup"] = len(deduped)

# =========================
# Split train/valid/test
# =========================
rng = random.Random(SEED)
rng.shuffle(deduped)

n = len(deduped)
n_train = int(n * TRAIN_RATIO)
n_valid = int(n * VALID_RATIO)
n_test  = n - n_train - n_valid

train = deduped[:n_train]
valid = deduped[n_train:n_train+n_valid]
test  = deduped[n_train+n_valid:]

def write_parallel(pairs, out_prefix):
    ar_path = os.path.join(OUT_DIR, f"{out_prefix}.ar")
    ig_path = os.path.join(OUT_DIR, f"{out_prefix}.ig")
    with open(ar_path, "w", encoding="utf-8") as fa, open(ig_path, "w", encoding="utf-8") as fi:
        for a, i in pairs:
            fa.write(a + "\n")
            fi.write(i + "\n")
    return ar_path, ig_path

train_paths = write_parallel(train, "train")
valid_paths = write_parallel(valid, "valid")
test_paths  = write_parallel(test,  "test")

# Also save full deduped corpus (useful for SentencePiece training)
_ = write_parallel(deduped, "all")

report = {
    "input": {
        "arabic_chapter_files": len(ar_files),
        "igbo_chapter_files": len(ig_files),
        "common_keys": len(common_keys),
    },
    "alignment": {
        "raw_mismatched_chapters": len(mismatches),
        "repaired_mismatch_chapters": repaired,
        "mismatch_examples_first10": mismatches[:10],
    },
    "filters_option_A": {
        "min_ar_chars": MIN_AR_CHARS,
        "min_words_either_side": MIN_WORDS_EITHER_SIDE,
    },
    "counts": dict(stats),
    "split": {
        "seed": SEED,
        "total_after_dedup": n,
        "train": len(train),
        "valid": len(valid),
        "test": len(test),
        "ratios": {"train": TRAIN_RATIO, "valid": VALID_RATIO, "test": TEST_RATIO},
    },
    "outputs": {
        "train_ar": train_paths[0], "train_ig": train_paths[1],
        "valid_ar": valid_paths[0], "valid_ig": valid_paths[1],
        "test_ar":  test_paths[0],  "test_ig":  test_paths[1],
        "all_ar": os.path.join(OUT_DIR, "all.ar"),
        "all_ig": os.path.join(OUT_DIR, "all.ig"),
    }
}

with open(os.path.join(OUT_DIR, "preprocess_report.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("✅ Done.")
print(json.dumps(report["split"], indent=2))
print("Saved to:", OUT_DIR)


In [ ]:

"""
import random
import os

SEED = 42
random.seed(SEED)

OUTPUT_DIR = "/content/drive/MyDrive/translation_data/splits"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def split_parallel_files(ar_path, ig_path):
    with open(ar_path, encoding="utf-8") as fa, open(ig_path, encoding="utf-8") as fi:
        ar_lines = fa.readlines()
        ig_lines = fi.readlines()

    assert len(ar_lines) == len(ig_lines)

    data = list(zip(ar_lines, ig_lines))
    random.shuffle(data)

    n = len(data)
    train = data[:int(0.8 * n)]
    dev   = data[int(0.8 * n):int(0.9 * n)]
    test  = data[int(0.9 * n):]

    def write_split(name, pairs):
        with open(f"{OUTPUT_DIR}/{name}.ar", "w", encoding="utf-8") as fa, \
             open(f"{OUTPUT_DIR}/{name}.ibo", "w", encoding="utf-8") as fi:
            for a, i in pairs:
                fa.write(a)
                fi.write(i)

    write_split("train", train)
    write_split("valid", dev)
    write_split("test", test)

split_parallel_files(
    "/content/drive/MyDrive/translation_data/train.ar",
    "/content/drive/MyDrive/translation_data/train.ig"
)
"""

In [ ]:
import random

# ===== Paths (edit if needed) =====
train_ar_path  = "/content/drive/MyDrive/translation_data/processed_lenfilter/train.ar"
train_ig_path  = "/content/drive/MyDrive/translation_data/processed_lenfilter/train.ig"

valid_ar_path  = "/content/drive/MyDrive/translation_data/processed_lenfilter/valid.ar"
valid_ig_path  = "/content/drive/MyDrive/translation_data/processed_lenfilter/valid.ig"

test_ar_path   = "/content/drive/MyDrive/translation_data/processed_lenfilter/test.ar"
test_ig_path   = "/content/drive/MyDrive/translation_data/processed_lenfilter/test.ig"

# ===== Helper =====
def read_lines(path):
    with open(path, encoding="utf-8") as f:
        return [l.rstrip("\n") for l in f]

def preview_split(name, ar_path, ig_path, n_first=5, n_random=5, seed=42):
    ar_lines = read_lines(ar_path)
    ig_lines = read_lines(ig_path)

    print(f"\n===== {name.upper()} =====")
    print(f"Arabic lines: {len(ar_lines)} | Igbo lines: {len(ig_lines)}")

    if len(ar_lines) != len(ig_lines):
        print("❌ MISMATCH: different number of lines!")
        return

    # First n pairs
    print(f"\n🔹 First {n_first} pairs:")
    for i in range(min(n_first, len(ar_lines))):
        print(f"#{i+1}")
        print("AR :", ar_lines[i])
        print("IG :", ig_lines[i])
        print("----")

    # Random n pairs
    rng = random.Random(seed)
    idxs = rng.sample(range(len(ar_lines)), k=min(n_random, len(ar_lines)))

    print(f"\n🎲 Random {len(idxs)} pairs:")
    for idx in idxs:
        print(f"#{idx+1}")
        print("AR :", ar_lines[idx])
        print("IG :", ig_lines[idx])
        print("----")

# ===== Run previews =====
preview_split("train", train_ar_path, train_ig_path, n_first=5, n_random=5, seed=42)
preview_split("valid", valid_ar_path, valid_ig_path, n_first=5, n_random=5, seed=42)
preview_split("test",  test_ar_path,  test_ig_path,  n_first=5, n_random=5, seed=42)


In [ ]:
#models for the m2m100
"""
!wget -qq "https://dl.fbaipublicfiles.com/m2m_100/spm.128k.model"
!wget -qq "https://dl.fbaipublicfiles.com/m2m_100/data_dict.128k.txt"
!wget -qq "https://dl.fbaipublicfiles.com/m2m_100/model_dict.128k.txt"
!wget -qq "https://dl.fbaipublicfiles.com/m2m_100/language_pairs_small_models.txt"
!wget     "https://dl.fbaipublicfiles.com/m2m_100/418M_last_checkpoint.pt"
"""

In [ ]:
# ✅ Full cell (fixed): M2M100 uses "ig" not "ibo"
!pip -q install datasets transformers sentencepiece

from datasets import Dataset
from transformers import M2M100Tokenizer

# -------------------------
# Paths (edit if needed)
# -------------------------
train_ar_path  = "/content/drive/MyDrive/translation_data/processed_lenfilter/train.ar"
train_ig_path  = "/content/drive/MyDrive/translation_data/processed_lenfilter/train.ig"

valid_ar_path  = "/content/drive/MyDrive/translation_data/processed_lenfilter/valid.ar"
valid_ig_path  = "/content/drive/MyDrive/translation_data/processed_lenfilter/valid.ig"

test_ar_path   = "/content/drive/MyDrive/translation_data/processed_lenfilter/test.ar"
test_ig_path   = "/content/drive/MyDrive/translation_data/processed_lenfilter/test.ig"

# -------------------------
# Load raw splits
# -------------------------
def load_split(ar_path, ig_path):
    with open(ar_path, encoding="utf-8") as fa, open(ig_path, encoding="utf-8") as fi:
        ar_lines = [l.strip() for l in fa if l.strip()]
        ig_lines = [l.strip() for l in fi if l.strip()]

    assert len(ar_lines) == len(ig_lines), f"Line mismatch: {ar_path} vs {ig_path}"
    return Dataset.from_dict({"ar": ar_lines, "ibo": ig_lines})  # column name can stay "ibo"

train_dataset = load_split(train_ar_path, train_ig_path)
valid_dataset = load_split(valid_ar_path, valid_ig_path)
test_dataset  = load_split(test_ar_path,  test_ig_path)

print("Loaded:")
print("  train:", len(train_dataset))
print("  valid:", len(valid_dataset))
print("  test :", len(test_dataset))

# -------------------------
# Tokenizer (M2M100)
# -------------------------
tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")

# Show possible Igbo codes in this tokenizer
ig_candidates = [k for k in tokenizer.lang_code_to_token.keys() if "ig" in k.lower()]
print("\nLanguage codes containing 'ig':", ig_candidates)

# ✅ Correct codes for M2M100:
tokenizer.src_lang = "ar"
tokenizer.tgt_lang = "ig"   # NOT "ibo"

MAX_LEN = 256  # use 128 if you want faster/smaller

def tokenize(batch):
    # This API uses the tokenizer's src_lang/tgt_lang internally
    return tokenizer(
        batch["ar"],
        text_target=batch["ibo"],
        truncation=True,
        max_length=MAX_LEN
    )

# Tokenize (no fixed padding here; Trainer will pad dynamically later)
train_tok = train_dataset.map(tokenize, batched=True, remove_columns=["ar", "ibo"])
valid_tok = valid_dataset.map(tokenize, batched=True, remove_columns=["ar", "ibo"])
test_tok  = test_dataset.map(tokenize,  batched=True, remove_columns=["ar", "ibo"])

print("\nTokenized columns:", train_tok.column_names)
print("Example input_ids length:", len(train_tok[0]["input_ids"]))
print("Example labels length   :", len(train_tok[0]["labels"]))
print("\n✅ Tokenization done. Next step: model + DataCollatorForSeq2Seq + Seq2SeqTrainer.")


fine tuning

In [ ]:
!pip -q install -U wandb

import os, wandb
wandb.login()

# choose a project name
os.environ["WANDB_PROJECT"] = "ar-ig-mt-m2m100"
# optional: if you have a team/entity
# os.environ["WANDB_ENTITY"] = "YOUR_ENTITY"


In [ ]:
!pip -q install -U transformers accelerate sentencepiece datasets evaluate sacrebleu


In [ ]:
import os, json, datetime
import numpy as np
import evaluate
import transformers
from inspect import signature
from transformers import (
    M2M100ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    GenerationConfig
)

print("Transformers:", transformers.__version__)

# ---- Metrics loaders ----
sacrebleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")

def bleu_chrf(preds, labels):
    pred_text = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    ref_text = tokenizer.batch_decode(labels, skip_special_tokens=True)

    pred_text = [p.strip() for p in pred_text]
    ref_text  = [[r.strip()] for r in ref_text]

    bleu = sacrebleu.compute(predictions=pred_text, references=ref_text)["score"]
    c    = chrf.compute(predictions=pred_text, references=ref_text)["score"]
    return float(bleu), float(c)

# ---- Version-safe TrainingArguments builder ----
def make_args(output_dir, run_name, lr, epochs, bs, grad_accum, eval_steps=2000, save_steps=2000):
    params = set(signature(Seq2SeqTrainingArguments.__init__).parameters.keys())
    kwargs = dict(
        output_dir=output_dir,
        learning_rate=lr,
        num_train_epochs=epochs,
        per_device_train_batch_size=bs,
        per_device_eval_batch_size=bs,
        gradient_accumulation_steps=grad_accum,
        logging_steps=50,
        eval_steps=eval_steps,
        save_steps=save_steps,
        save_total_limit=2,
        fp16=True,
        predict_with_generate=False,   # fast during training
        dataloader_num_workers=2,
        report_to="wandb",             # ✅ W&B ON (was "none" in your code) :contentReference[oaicite:2]{index=2}
        run_name=run_name,
    )
    if "evaluation_strategy" in params:
        kwargs["evaluation_strategy"] = "steps"
    elif "eval_strategy" in params:
        kwargs["eval_strategy"] = "steps"
    if "save_strategy" in params:
        kwargs["save_strategy"] = "steps"
    return Seq2SeqTrainingArguments(**kwargs)

# ---- One run function ----
def run_one(cfg):
    run_name = cfg["run_name"]
    out_dir = f"/content/runs/{run_name}"
    os.makedirs(out_dir, exist_ok=True)

    wandb.init(name=run_name, config=cfg)  # ✅ each run is tracked

    # fresh model each run (fair comparison)
    model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M")

    # force target language (Igbo code for M2M100 is "ig")
    model.generation_config = GenerationConfig.from_model_config(model.config)
    model.generation_config.forced_bos_token_id = tokenizer.get_lang_id("ig")
    model.generation_config.max_length = 256
    model.generation_config.num_beams = 5
    model.generation_config.early_stopping = True

    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

    args = make_args(
        output_dir=out_dir,
        run_name=run_name,
        lr=cfg["lr"],
        epochs=cfg["epochs"],
        bs=cfg["bs"],
        grad_accum=cfg["grad_accum"],
        eval_steps=cfg.get("eval_steps", 2000),
        save_steps=cfg.get("save_steps", 2000),
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=valid_tok,
        data_collator=data_collator,
        # no need to pass tokenizer (deprecated warning in your logs)
    )

    trainer.train()

    # ---- Evaluate with generation AFTER training ----
    trainer.args.predict_with_generate = True

    pv = trainer.predict(valid_tok)
    bleu_v, chrf_v = bleu_chrf(pv.predictions, pv.label_ids)

    pt = trainer.predict(test_tok)
    bleu_t, chrf_t = bleu_chrf(pt.predictions, pt.label_ids)

    wandb.log({
        "eval/valid_sacrebleu": bleu_v,
        "eval/valid_chrf": chrf_v,
        "eval/test_sacrebleu": bleu_t,
        "eval/test_chrf": chrf_t,
    })

    print(f"\n[{run_name}] VALID BLEU {bleu_v:.2f} | chrF {chrf_v:.2f}")
    print(f"[{run_name}] TEST  BLEU {bleu_t:.2f} | chrF {chrf_t:.2f}")

    # ---- Save best artifacts to Drive (optional but recommended) ----
    drive_out = f"/content/drive/MyDrive/translation_data/runs/{run_name}"
    os.makedirs(drive_out, exist_ok=True)
    trainer.save_model(drive_out)
    tokenizer.save_pretrained(drive_out)

    # ---- Save a small JSON summary to Drive (for thesis) ----
    summary_dir = "/content/drive/MyDrive/translation_data/experiments"
    os.makedirs(summary_dir, exist_ok=True)
    summary_path = os.path.join(summary_dir, f"{run_name}_summary.json")

    summary = {
        "timestamp": datetime.datetime.now().isoformat(),
        "run_name": run_name,
        "config": cfg,
        "metrics": {
            "valid": {"sacrebleu": bleu_v, "chrf": chrf_v},
            "test":  {"sacrebleu": bleu_t, "chrf": chrf_t},
        },
        "saved_model_dir": drive_out
    }
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("✅ Saved model to:", drive_out)
    print("✅ Saved summary to:", summary_path)

    wandb.finish()
    return summary

# ---- 1 baseline + 1–2 comparisons (minimal sweep) ----
runs = [
    # Baseline (same as your current one) :contentReference[oaicite:3]{index=3}
    {"run_name": "baseline_lr3e-5_ep3_bs8_ga2", "lr": 3e-5, "epochs": 3, "bs": 8, "grad_accum": 2},

    # Comparison #1: lower LR
    {"run_name": "lr2e-5_ep3_bs8_ga2",          "lr": 2e-5, "epochs": 3, "bs": 8, "grad_accum": 2},

    # Comparison #2 (optional): more epochs
     #{"run_name": "lr3e-5_ep5_bs8_ga2",        "lr": 3e-5, "epochs": 5, "bs": 8, "grad_accum": 2},
]

summaries = [run_one(cfg) for cfg in runs]

# pick best by VALID BLEU
best = max(summaries, key=lambda s: s["metrics"]["valid"]["sacrebleu"])
print("\n🏆 Best run by VALID BLEU:", best["run_name"], best["metrics"])


In [ ]:
# ✅ Summarize all experiment JSONs in your Drive folder into a table + save CSV

import os, json
import pandas as pd

EXP_DIR = "/content/drive/MyDrive/translation_data/experiments"
OUT_CSV = "/content/drive/MyDrive/translation_data/experiments/runs_summary.csv"

rows = []
for fn in sorted(os.listdir(EXP_DIR)):
    if not fn.endswith("_summary.json"):
        continue
    path = os.path.join(EXP_DIR, fn)
    with open(path, "r", encoding="utf-8") as f:
        s = json.load(f)

    cfg = s.get("config", {})
    metrics = s.get("metrics", {})
    v = metrics.get("valid", {})
    t = metrics.get("test", {})

    rows.append({
        "run_name": s.get("run_name", fn.replace("_summary.json", "")),
        "timestamp": s.get("timestamp", ""),
        "lr": cfg.get("lr", cfg.get("learning_rate", None)),
        "epochs": cfg.get("epochs", cfg.get("num_train_epochs", None)),
        "bs": cfg.get("bs", cfg.get("per_device_train_batch_size", None)),
        "grad_accum": cfg.get("grad_accum", cfg.get("gradient_accumulation_steps", None)),
        "valid_bleu": v.get("sacrebleu", None),
        "valid_chrf": v.get("chrf", None),
        "test_bleu": t.get("sacrebleu", None),
        "test_chrf": t.get("chrf", None),
        "saved_model_dir": s.get("saved_model_dir", ""),
        "summary_path": path
    })

df = pd.DataFrame(rows)

if len(df) == 0:
    print("No *_summary.json files found in:", EXP_DIR)
else:
    # Sort by best VALID BLEU (primary), then TEST BLEU (secondary)
    df_sorted = df.sort_values(by=["valid_bleu", "test_bleu"], ascending=False).reset_index(drop=True)

    print("✅ Found", len(df_sorted), "runs. Best run:")
    print(df_sorted.loc[0, ["run_name", "valid_bleu", "valid_chrf", "test_bleu", "test_chrf", "saved_model_dir"]])

    display(df_sorted[[
        "run_name","lr","epochs","bs","grad_accum",
        "valid_bleu","valid_chrf","test_bleu","test_chrf",
        "saved_model_dir"
    ]])

    df_sorted.to_csv(OUT_CSV, index=False)
    print("\n✅ Saved CSV to:", OUT_CSV)


In [ ]:
EXP_DIR  = "/content/drive/MyDrive/translation_data/experiments"
RUNS_DIR = "/content/drive/MyDrive/translation_data/runs"

print("=== experiments folder ===")
!ls -lah "{EXP_DIR}"

print("\n=== runs folder ===")
!ls -lah "{RUNS_DIR}"


In [ ]:
# ✅ Evaluation cell (loads fine-tuned checkpoint from Drive + computes SacreBLEU & chrF)
"""
!pip -q install -U transformers accelerate sentencepiece datasets evaluate sacrebleu

import os, re
import numpy as np
import evaluate
from datasets import Dataset
from transformers import (
    M2M100Tokenizer,
    M2M100ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    GenerationConfig
)

# ---- 1) Load model from your Drive folder (auto-pick latest checkpoint if needed) ----
MODEL_ROOT = "/content/drive/MyDrive/translation_data/m2m100-finetuned"

def resolve_model_dir(root):
    if os.path.exists(os.path.join(root, "config.json")):
        return root
    ckpts = []
    for name in os.listdir(root):
        if name.startswith("checkpoint-"):
            m = re.search(r"checkpoint-(\d+)", name)
            if m:
                ckpts.append((int(m.group(1)), os.path.join(root, name)))
    if not ckpts:
        raise FileNotFoundError(f"No config.json in {root} and no checkpoint-* found.")
    ckpts.sort(key=lambda x: x[0])
    return ckpts[-1][1]

MODEL_DIR = resolve_model_dir(MODEL_ROOT)
print("✅ Loading from:", MODEL_DIR)

# tokenizer might not exist inside checkpoint -> fallback
try:
    tokenizer = M2M100Tokenizer.from_pretrained(MODEL_DIR)
except Exception:
    tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")

model = M2M100ForConditionalGeneration.from_pretrained(MODEL_DIR)

# M2M100 Igbo code is "ig"
tokenizer.src_lang = "ar"
tokenizer.tgt_lang = "ig"

model.generation_config = GenerationConfig.from_model_config(model.config)
model.generation_config.forced_bos_token_id = tokenizer.get_lang_id("ig")

# ---- 2) Load your eval splits ----
DATA_DIR = "/content/drive/MyDrive/translation_data/processed_lenfilter"  # change if needed

def load_split(ar_path, ig_path):
    with open(ar_path, encoding="utf-8") as fa, open(ig_path, encoding="utf-8") as fi:
        ar = [l.strip() for l in fa if l.strip()]
        ig = [l.strip() for l in fi if l.strip()]
    assert len(ar) == len(ig), "Line mismatch!"
    return Dataset.from_dict({"ar": ar, "ig": ig})

valid_ds = load_split(f"{DATA_DIR}/valid.ar", f"{DATA_DIR}/valid.ig")
test_ds  = load_split(f"{DATA_DIR}/test.ar",  f"{DATA_DIR}/test.ig")

# ---- 3) Tokenize ----
MAX_LEN = 256

def tokenize(batch):
    x = tokenizer(batch["ar"], truncation=True, max_length=MAX_LEN)
    with tokenizer.as_target_tokenizer():
        y = tokenizer(batch["ig"], truncation=True, max_length=MAX_LEN)
    x["labels"] = y["input_ids"]
    return x

valid_tok = valid_ds.map(tokenize, batched=True, remove_columns=["ar","ig"])
test_tok  = test_ds.map(tokenize,  batched=True, remove_columns=["ar","ig"])

# ---- 4) Predict + metrics ----
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
args = Seq2SeqTrainingArguments(
    output_dir="/content/tmp_eval",
    predict_with_generate=True,
    report_to="none",
    per_device_eval_batch_size=8
)

trainer = Seq2SeqTrainer(model=model, args=args, data_collator=data_collator)

sacrebleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")

def bleu_chrf(preds, labels):
    pred_text = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    ref_text = tokenizer.batch_decode(labels, skip_special_tokens=True)
    pred_text = [p.strip() for p in pred_text]
    ref_text = [[r.strip()] for r in ref_text]
    bleu = sacrebleu.compute(predictions=pred_text, references=ref_text)["score"]
    c = chrf.compute(predictions=pred_text, references=ref_text)["score"]
    return bleu, c
model.generation_config.early_stopping = True
model.generation_config.max_length = 256
model.generation_config.num_beams = 5
model.generation_config.early_stopping = True

pv = trainer.predict(valid_tok, max_length=256, num_beams=5)
bleu_v, chrf_v = bleu_chrf(pv.predictions, pv.label_ids)
print(f"VALID  SacreBLEU: {bleu_v:.2f} | chrF: {chrf_v:.2f}")

pt = trainer.predict(test_tok, max_length=256, num_beams=5)
bleu_t, chrf_t = bleu_chrf(pt.predictions, pt.label_ids)
print(f"TEST   SacreBLEU: {bleu_t:.2f} | chrF: {chrf_t:.2f}")
"""


In [ ]:
"""
import json, datetime, os

OUT_PATH = "/content/drive/MyDrive/translation_data/experiments/m2m100_ar_ig_eval.json"
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

record = {
    "timestamp": datetime.datetime.now().isoformat(),
    "task": "Arabic -> Igbo",
    "model": "facebook/m2m100_418M (fine-tuned)",
    "data_dir": "/content/drive/MyDrive/translation_data/processed_lenfilter",
    "split_sizes": {
        "train": 30337,
        "valid": 309,
        "test": 311
    },
    "generation": {
        "num_beams": 5,
        "max_length": 256,
        "forced_bos_lang": "ig"
    },
    "metrics": {
        "valid": {"sacrebleu": 19.32, "chrf": 43.37},
        "test":  {"sacrebleu": 20.36, "chrf": 42.51}
    }
}

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(record, f, ensure_ascii=False, indent=2)

print("✅ Saved evaluation record to:", OUT_PATH)
